# Principal Component Analysis
Accompanies the notes on **Principal Component Analysis**.

## 1. Four points by hand

In [ ]:
import numpy as np

X = np.array([[7, 6], [6, 7], [3, 4], [4, 3]], dtype=float)
Xc = X - X.mean(axis=0)
cov = Xc.T @ Xc / (len(X) - 1)
print("covariance matrix:\n", cov.round(4))
eigenvalues, eigenvectors = np.linalg.eigh(cov)            # eigh: for symmetric matrices, ascending order
order = np.argsort(eigenvalues)[::-1]
eigenvalues, eigenvectors = eigenvalues[order], eigenvectors[:, order]
print("eigenvalues:", eigenvalues.round(4), " explained variance ratio:", (eigenvalues / eigenvalues.sum()).round(3))
print("eigenvectors (columns):\n", eigenvectors.round(4))
Z = Xc @ eigenvectors
print("scores:\n", Z.round(3))
X_hat = Z[:, [0]] @ eigenvectors[:, [0]].T + X.mean(axis=0)      # keep only PC1, map back
print("reconstruction from PC1:\n", X_hat)
print("squared reconstruction error:", ((X - X_hat) ** 2).sum().round(4))

## 2. PCA from scratch, checked against scikit-learn

In [ ]:
import numpy as np
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

class FirstPrinciplesPCA:
    """
    PCA implementation from first principles using Eigendecomposition and SVD.
    """
    def __init__(self, n_components=None):
        self.n_components = n_components
        self.components_ = None          # Eigenvectors (U_k)
        self.explained_variance_ = None  # Eigenvalues (lambda)
        self.explained_variance_ratio_ = None
        self.mean_ = None
        
    def fit(self, X):
        n_samples, n_features = X.shape
        
        # 1. Compute mean and center the design matrix
        self.mean_ = np.mean(X, axis=0)
        X_centered = X - self.mean_
        
        # 2. Compute sample covariance matrix: Sigma = (X^T X) / (n - 1)
        cov_matrix = np.dot(X_centered.T, X_centered) / (n_samples - 1)
        
        # 3. Compute eigenvalues and eigenvectors
        eigenvalues, eigenvectors = np.linalg.eigh(cov_matrix)
        
        # 4. Sort eigenvalues and eigenvectors in descending order
        sorted_indices = np.argsort(eigenvalues)[::-1]
        eigenvalues = eigenvalues[sorted_indices]
        eigenvectors = eigenvectors[:, sorted_indices]
        
        # Calculate total variance and explained variance ratio
        total_variance = np.sum(eigenvalues)
        self.explained_variance_ = eigenvalues
        self.explained_variance_ratio_ = eigenvalues / total_variance
        
        # 5. Select top k components
        if self.n_components is not None:
            if isinstance(self.n_components, float) and 0.0 < self.n_components < 1.0:
                cumulative_evr = np.cumsum(self.explained_variance_ratio_)
                k = np.argmax(cumulative_evr >= self.n_components) + 1
            else:
                k = int(self.n_components)
        else:
            k = n_features
            
        self.components_ = eigenvectors[:, :k].T  # Format as (k, p) to match sklearn
        self.explained_variance_ = self.explained_variance_[:k]
        self.explained_variance_ratio_ = self.explained_variance_ratio_[:k]
        return self

    def transform(self, X):
        X_centered = X - self.mean_
        # Projected coordinates: Z = X_tilde * U_k
        return np.dot(X_centered, self.components_.T)

    def inverse_transform(self, Z):
        # Reconstruction: X_hat = Z * U_k^T + mean
        return np.dot(Z, self.components_) + self.mean_

# check against scikit-learn on correlated 3-D data
rng = np.random.default_rng(42)
n = 500
x1 = rng.normal(0, 3, n)
x2 = 0.8 * x1 + rng.normal(0, 1, n)
x3 = 0.5 * x1 - 0.3 * x2 + rng.normal(0, 0.5, n)
X3 = np.column_stack([x1, x2, x3])
mine = FirstPrinciplesPCA(n_components=2).fit(X3)
skl = PCA(n_components=2).fit(X3)
print("ours:        ", mine.explained_variance_ratio_.round(6))
print("scikit-learn:", skl.explained_variance_ratio_.round(6))
# components can differ by a sign, which describes the same axis
print("same axes:", np.allclose(np.abs(mine.components_), np.abs(skl.components_)))

## 3. Why we scale: bank columns unscaled

In [ ]:
import pandas as pd
bd = pd.read_csv("../../data/bd_train.csv")
cols = ["Average.Credit.Card.Transaction", "Balance.Transfer", "Term.Deposit", "Life.Insurance",
        "Medical.Insurance", "Average.A.C.Balance", "Personal.Loan", "Investment.in.Mutual.Fund",
        "Investment.Tax.Saving.Bond", "Home.Loan", "Online.Purchase.Amount", "Investment.in.Commudity",
        "Investment.in.Equity", "Investment.in.Derivative", "Portfolio.Balance"]
raw = bd[cols]
print("largest column variances:\n", raw.var().sort_values(ascending=False).head(3).round(0))
pca_raw = PCA().fit(raw)
print("raw: PC1 explains", pca_raw.explained_variance_ratio_[0].round(3),
      "| biggest PC1 weight:", cols[np.abs(pca_raw.components_[0]).argmax()],
      np.abs(pca_raw.components_[0]).max().round(3))

## 4. PCA on the bank customers: scree plot, loadings, 2-D picture

In [ ]:
import matplotlib.pyplot as plt
logx = np.log1p(raw.clip(lower=0))
xs = (logx - logx.mean()) / logx.std(ddof=0)
pca = PCA().fit(xs)
print("explained variance ratio:", pca.explained_variance_ratio_.round(3))
print("cumulative:              ", pca.explained_variance_ratio_.cumsum().round(3))
loadings = pd.DataFrame(pca.components_[:2].T, index=cols, columns=["PC1", "PC2"]).round(2)
print(loadings.sort_values("PC2"))
Z = pca.transform(xs)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].bar(range(1, 16), pca.explained_variance_ratio_); axes[0].plot(range(1, 16), pca.explained_variance_ratio_.cumsum(), "o-", color="crimson")
axes[0].set_xlabel("component"); axes[0].set_title("explained variance ratio (bars), cumulative (line)")
hi = bd["Revenue.Grid"] == 1
axes[1].scatter(Z[~hi, 0], Z[~hi, 1], s=3, alpha=0.3, label="Revenue.Grid = 2")
axes[1].scatter(Z[hi, 0], Z[hi, 1], s=3, alpha=0.6, color="crimson", label="Revenue.Grid = 1")
axes[1].set_xlabel("PC1"); axes[1].set_ylabel("PC2"); axes[1].legend(markerscale=4); axes[1].set_title("customers on the first two components")
plt.tight_layout(); plt.show()

## 5. PCA before a model?

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

data = bd.drop(columns=["REF_NO", "post_code", "post_area"])
y = (data.pop("Revenue.Grid") == 1).astype(int)
Xall = pd.get_dummies(data, drop_first=True, dtype=float)
x_train, x_test, y_train, y_test = train_test_split(Xall, y, test_size=0.25, random_state=1, stratify=y)
for k in [2, 5, 10, 20, 40, 60, None]:
    steps = [("scale", StandardScaler())] + ([("pca", PCA(n_components=k))] if k else []) + [("lr", LogisticRegression(max_iter=5000))]
    pipe = Pipeline(steps).fit(x_train, y_train)
    print(f"{'all 97 features' if k is None else str(k) + ' components':16s}: test AUC {roc_auc_score(y_test, pipe.predict_proba(x_test)[:, 1]):.4f}")

## Try it yourself
1. Four points (7,4),(6,3),(3,6),(4,7) in section 1.
2. Components needed for 80% variance, unscaled vs standardised.
3. Plot PC1 vs PC3 and name PC3 from its loadings.